# S16 · A 10-stock NSE portfolio, start to finish

We put everything together on a basket of ten real NSE stocks: build the efficient
frontier, find the best reward-per-risk mix, show its weights as a bar chart, and
report its return, risk and Sharpe ratio. This is exactly the deliverable a quant
analyst hands over: how much of your ₹1 lakh goes into each stock.

**New here? Read this once.**

- New to Python? Press play on each cell top to bottom and read the note above
  each one. You do not need to write any code.
- Want the ideas first? The primer `primers/diversification_and_covariance.md`
  covers return, risk and covariance in ten minutes.
- Already confident? Look for the cell marked **Stretch (optional)** near the end,
  where we add a rule that spreads the money out more.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses cvxpy and yfinance, which Colab does not ship reliably.
import sys
if "google.colab" in sys.modules:
    !pip install -q cvxpy yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — get our tools ready

The same tools as notebook 2: NumPy, pandas, matplotlib, and CVXPY for the
optimisation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

np.random.seed(0)
print("Tools ready.")

## Step 2 — get prices for ten NSE stocks

We use ten well-known Indian companies from different sectors: software, banking,
energy, consumer goods, cars, pharma, engineering, metals. Mixing sectors is
deliberate, because companies in different sectors move together less, which is
exactly what makes diversification work. Same helper as before: live download, or a
synthetic fallback offline.

In [ ]:
# We try to download REAL daily prices from Yahoo Finance.
# If there is no internet, we instead build a synthetic - made up but realistic -
# set of prices, so this notebook ALWAYS runs. We print which one happened.

# Ten NSE stocks from different sectors. '.NS' = National Stock Exchange of India.
stock_tickers = ["TCS.NS", "INFY.NS", "HDFCBANK.NS", "RELIANCE.NS", "ITC.NS",
                 "MARUTI.NS", "SUNPHARMA.NS", "LT.NS", "NESTLEIND.NS", "TATASTEEL.NS"]

used_live_data = False   # we will flip this to True if the download works

try:
    import yfinance as yf

    # Download about three years of daily prices for all the stocks at once.
    raw = yf.download(stock_tickers, period="3y", auto_adjust=True, progress=False)

    # Keep the 'Close' price - the price at the end of each trading day.
    prices = raw["Close"]

    # Drop any days where a price is missing, and keep the columns in our order.
    prices = prices.dropna()
    prices = prices[stock_tickers]

    if len(prices) < 50:
        raise ValueError("not enough price data came back")

    used_live_data = True
    print("Downloaded LIVE price data from Yahoo Finance.")

except Exception as problem:
    # ---- Offline fallback: build synthetic prices with NumPy. -------------
    print("Could not download live data, so we are using SYNTHETIC data instead.")
    print("(Reason:", problem, ")")

    number_of_stocks = len(stock_tickers)
    number_of_days = 750

    # Each stock has its own average daily return and its own wobble (risk).
    yearly_return = np.random.uniform(0.06, 0.20, size=number_of_stocks)
    yearly_risk = np.random.uniform(0.18, 0.40, size=number_of_stocks)
    daily_average_return = yearly_return / 252.0      # ~252 trading days a year
    daily_risk = yearly_risk / np.sqrt(252.0)

    # Mix a few shared 'market mood' signals into each stock so they move together.
    number_of_shared_factors = 3
    loadings = np.random.uniform(0.2, 1.0, size=(number_of_stocks, number_of_shared_factors))
    shared_moods = np.random.normal(0, 1, size=(number_of_days, number_of_shared_factors))
    own_wiggle = np.random.normal(0, 1, size=(number_of_days, number_of_stocks))

    daily_returns_matrix = shared_moods @ loadings.T + own_wiggle

    # Scale each column so it has the daily average return and risk we chose.
    for stock_number in range(number_of_stocks):
        column = daily_returns_matrix[:, stock_number]
        column = (column - column.mean()) / column.std()      # standardise
        column = column * daily_risk[stock_number] + daily_average_return[stock_number]
        daily_returns_matrix[:, stock_number] = column

    price_paths = 100 * np.cumprod(1 + daily_returns_matrix, axis=0)
    business_days = pd.bdate_range(end=pd.Timestamp.today(), periods=number_of_days)
    prices = pd.DataFrame(price_paths, index=business_days, columns=stock_tickers)

print("\nPrice table shape (rows = days, columns = stocks):", prices.shape)
prices.head()

## Step 3 — expected returns μ and covariance Σ

Exactly as before: average daily returns for μ, covariance for Σ, both annualised.
We also run the quick health check on Σ (its smallest eigenvalue should be zero or
positive) so we know every mix will have a sensible, non-negative risk.

In [ ]:
trading_days_per_year = 252
daily_returns = prices.pct_change().dropna()
# Expected yearly return of each stock, and a table of how the stocks move together.
mu = (daily_returns.mean() * trading_days_per_year).values
Sigma = (daily_returns.cov() * trading_days_per_year).values
number_of_stocks = len(mu)
stock_names = list(prices.columns)
print("Number of stocks:", number_of_stocks)
print("Return and risk tables are ready. A portfolio's risk can never come out")
print("negative, so we are good to go.")

## Step 4 — the frontier function

The same little function from notebook 2. For a given target return, it finds the
weights that give the least risk, with no short selling.

In [ ]:
def best_weights_for_target(target):
    w = cp.Variable(number_of_stocks)
    constraints = [
        cp.sum(w) == 1,           # invest all the money
        w >= 0,                   # no short selling
        mu @ w >= target,         # reach at least the target return
    ]
    problem = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), constraints)
    problem.solve()
    if w.value is None:
        return None
    return w.value

print("Frontier function ready.")

## Step 5 — build the efficient frontier

Sweep the target return across its full range and record the risk, return and
weights of each best mix. We keep the weights this time, because at the end we want
the actual recipe.

In [ ]:
target_returns = np.linspace(mu.min(), mu.max(), 50)

frontier_risk = []
frontier_return = []
frontier_weights = []     # we also keep the weights this time

for target in target_returns:
    weights = best_weights_for_target(target)
    if weights is None:
        continue
    frontier_weights.append(weights)
    frontier_return.append(weights @ mu)
    frontier_risk.append(np.sqrt(weights @ Sigma @ weights))

frontier_risk = np.array(frontier_risk)
frontier_return = np.array(frontier_return)

print("Computed", len(frontier_risk), "portfolios along the frontier.")

## Step 6 — find the maximum-Sharpe portfolio

Pick the point on the frontier with the best reward per unit of risk (the highest
Sharpe ratio). These are the weights a portfolio manager would actually act on.

In [ ]:
risk_free_rate = 0.06     # safe yearly return (6%)

frontier_sharpe = (frontier_return - risk_free_rate) / frontier_risk
best_position = np.argmax(frontier_sharpe)

best_weights = frontier_weights[best_position]
best_return = frontier_return[best_position]
best_risk = frontier_risk[best_position]
best_sharpe = frontier_sharpe[best_position]

print("Maximum-Sharpe portfolio found.")

## Step 7 — plot the frontier and mark the best portfolio

The full efficient frontier for the ten-stock basket, with the maximum-Sharpe
portfolio marked as a star.

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(frontier_risk * 100, frontier_return * 100,
         color="#2E75B6", linewidth=3, label="efficient frontier")

plt.scatter(best_risk * 100, best_return * 100,
            color="#F1C40F", s=300, marker="*",
            edgecolor="black", zorder=5, label="maximum Sharpe")

plt.xlabel("risk (volatility, %)")
plt.ylabel("expected return (%)")
plt.title("Efficient frontier for the 10-stock NSE basket")
plt.legend()
plt.show()

## Step 8 — show the winning mix as a bar chart

How much of our ₹1 lakh goes into each stock? We sort the weights from largest to
smallest and draw a bar chart. Notice the optimiser often piles into just a few
names. That is normal, and in practice we add a rule to spread it out more (the
Stretch cell at the end does exactly that).

In [ ]:
# Sort the stocks by weight, biggest first, so the chart is easy to read.
order = np.argsort(best_weights)[::-1]
sorted_names = [stock_names[i] for i in order]
sorted_weights = best_weights[order]

plt.figure(figsize=(10, 5))
plt.bar(sorted_names, sorted_weights * 100, color="#2E75B6", edgecolor="black")
plt.ylabel("weight (%)")
plt.title("Maximum-Sharpe portfolio: how much to put in each stock")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 9 — report the portfolio's numbers

Finally we print the three numbers a manager cares about: the expected yearly
return, the yearly risk, and the Sharpe ratio. We also print each stock's weight as
a tidy table.

In [ ]:
print("=== Maximum-Sharpe portfolio for the 10-stock NSE basket ===\n")
print("Expected yearly return:", round(best_return * 100, 2), "%")
print("Yearly risk (volatility):", round(best_risk * 100, 2), "%")
print("Sharpe ratio:", round(best_sharpe, 3))

print("\nWeights:")
# A small table of stock name and its weight, biggest first.
weights_table = pd.Series(best_weights, index=stock_names).sort_values(ascending=False)
print((weights_table * 100).round(2))

### Stretch (optional) — stop the optimiser over-concentrating

Skip this if you are new to code. The maximum-Sharpe mix above probably poured most
of the money into two or three names, which is risky in a different way: if you are
wrong about one of them, you are very wrong. A common fix is to add a rule saying no
single stock may hold more than, say, 25% of the money. In CVXPY that is one extra
line, `w <= 0.25`. We re-solve with that rule and compare the weights.

In [ ]:
# Real investment mandates often forbid putting more than a set fraction in any
# one stock. Let's add the rule "no single stock above 25%" and re-find the best
# (highest-Sharpe) mix under that cap. We build a capped frontier the same way as
# before; high-return targets are simply not reachable once no stock may exceed
# 25%, so those targets are skipped.
cap = 0.25

def best_capped_weights_for_target(target, cap):
    w = cp.Variable(number_of_stocks)
    constraints = [
        cp.sum(w) == 1,       # invest all the money
        w >= 0,               # no short selling
        w <= cap,             # NEW: no single stock above the cap
        mu @ w >= target,     # reach at least the target return
    ]
    problem = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), constraints)
    problem.solve()
    if w.value is None:       # this target is impossible under the cap
        return None
    return w.value

capped_returns = []
capped_risks = []
capped_weights_list = []
for target in target_returns:
    weights = best_capped_weights_for_target(target, cap)
    if weights is None:
        continue
    capped_weights_list.append(weights)
    capped_returns.append(weights @ mu)
    capped_risks.append(np.sqrt(weights @ Sigma @ weights))

capped_returns = np.array(capped_returns)
capped_risks = np.array(capped_risks)

# Pick the highest-Sharpe portfolio among the capped ones.
capped_sharpe = (capped_returns - risk_free_rate) / capped_risks
capped_best = np.argmax(capped_sharpe)
capped_weights = capped_weights_list[capped_best]

comparison = pd.DataFrame({
    "no cap (%)": (best_weights * 100).round(2),
    "capped at 25% (%)": (capped_weights * 100).round(2),
}, index=stock_names)

print("Forbidding any stock above 25% spreads the money across more names:\n")
print(comparison.sort_values("no cap (%)", ascending=False))
print("\nSharpe ratio  -  no cap:", round(best_sharpe, 3),
      "  capped:", round(capped_sharpe[capped_best], 3))

## What you just did

You took ten real NSE stocks, estimated their returns and the covariance table Σ,
checked the risk numbers were well behaved, and used CVXPY to build the efficient
frontier. You found the maximum-Sharpe mix, drew its weights as a bar chart, and
reported its return, risk and Sharpe ratio. That is portfolio optimisation, end to
end: given ₹1 lakh and ten stocks, here is a genuinely good way to split it.

### Stretch (optional) — a health check on the risk table

Skip unless curious. The risk table is well-formed only if every portfolio's risk comes
out non-negative. Mathematically that means its smallest **eigenvalue** is at least zero
(tiny negatives are just rounding).

In [ ]:
smallest = np.linalg.eigvalsh(Sigma).min()
print('smallest eigenvalue of the risk table:', round(float(smallest), 6))
print('at least zero (up to rounding) -> every portfolio risk is non-negative')